In [ ]:
from huggingface_hub import notebook_login

notebook_login()

In [ ]:
!pip install evaluate>=0.4.1
!pip install jiwer>=3.0.3

In [ ]:
"""
2_finetune.py
Fine-tune Whisper Small on the preprocessed LibriSpeech dataset.
Evaluates WER on the validation split after every epoch.
"""

import os
import torch
from dataclasses import dataclass
from typing import Any, Dict, List, Union
from datasets import load_dataset
from datasets import load_from_disk
from transformers import (
    WhisperForConditionalGeneration,
    WhisperProcessor,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
)
import evaluate

# ── Config ────────────────────────────────────────────────────────────────────
MODEL_ID     = "openai/whisper-small"
DATASET_ID   = "aminulpalash506/whisper-librispeech-processed"
OUTPUT_DIR   = "./whisper-small-librispeech"
LANGUAGE     = "english"
TASK         = "transcribe"
# ──────────────────────────────────────────────────────────────────────────────


# Training hyper-params (tiny values for demo; scale up for real training)
BATCH_SIZE       = 4      # per-device
GRAD_ACCUM_STEPS = 2      # effective batch = BATCH_SIZE * GRAD_ACCUM_STEPS
LEARNING_RATE    = 1e-5
WARMUP_STEPS     = 50
MAX_STEPS        = 200    # set -1 to train for full epochs instead
NUM_EPOCHS       = 3      # used only when MAX_STEPS == -1
FP16             = torch.cuda.is_available()
# ──────────────────────────────────────────────────────────────────────────────


# ── Data collator ─────────────────────────────────────────────────────────────
@dataclass
class DataCollatorSpeechSeq2SeqWithPadding:
    processor: Any

    def __call__(self, features: List[Dict[str, Union[List[int], torch.Tensor]]]) -> Dict[str, torch.Tensor]:
        # Pad input features (log-Mel spectrograms) to the same length
        input_features = [{"input_features": f["input_features"]} for f in features]
        batch = self.processor.feature_extractor.pad(input_features, return_tensors="pt")

        # Pad label sequences; replace padding token id with -100 so loss ignores them
        label_features = [{"input_ids": f["labels"]} for f in features]
        labels_batch   = self.processor.tokenizer.pad(label_features, return_tensors="pt")
        labels = labels_batch["input_ids"].masked_fill(labels_batch.attention_mask.ne(1), -100)

        # Strip BOS token if present (Whisper adds it during generation, not training)
        bos_token_id = self.processor.tokenizer.bos_token_id
        if (labels[:, 0] == bos_token_id).all():
            labels = labels[:, 1:]

        batch["labels"] = labels
        return batch
# ──────────────────────────────────────────────────────────────────────────────


def compute_metrics_fn(processor):
    """Return a compute_metrics function bound to the processor."""
    wer_metric = evaluate.load("wer")
    tokenizer  = processor.tokenizer

    def compute_metrics(pred):
        pred_ids   = pred.predictions
        label_ids  = pred.label_ids

        # Replace -100 back to pad token id so decode doesn't fail
        label_ids[label_ids == -100] = tokenizer.pad_token_id

        pred_str  = tokenizer.batch_decode(pred_ids,  skip_special_tokens=True)
        label_str = tokenizer.batch_decode(label_ids, skip_special_tokens=True)

        # Normalise: lower-case, strip leading/trailing spaces
        pred_str  = [p.lower().strip() for p in pred_str]
        label_str = [l.lower().strip() for l in label_str]

        wer = 100 * wer_metric.compute(predictions=pred_str, references=label_str)
        return {"wer": round(wer, 2)}

    return compute_metrics


def main():

    # ── Load dataset & processor ──────────────────────────────────────────────
    print("Loading dataset from Hugging Face Hub …")

    ds = load_dataset(DATASET_ID)

    processor = WhisperProcessor.from_pretrained(DATASET_ID)

    print(f"  Train size : {len(ds['train'])}")
    print(f"  Valid size : {len(ds['validation'])}")


    # ── Model ─────────────────────────────────────────────────────────────────
    print(f"Loading model '{MODEL_ID}' …")
    model = WhisperForConditionalGeneration.from_pretrained(MODEL_ID)

    model.generation_config.forced_decoder_ids = processor.get_decoder_prompt_ids(
    language=LANGUAGE, task=TASK
    )
    model.generation_config.suppress_tokens = []

    # Tell the model to always predict in English
    # model.config.forced_decoder_ids = processor.get_decoder_prompt_ids(
    #     language=LANGUAGE, task=TASK
    # )
    # model.config.suppress_tokens = []

    # ── Collator & metrics ────────────────────────────────────────────────────
    data_collator   = DataCollatorSpeechSeq2SeqWithPadding(processor=processor)
    compute_metrics = compute_metrics_fn(processor)

    # ── Training arguments ────────────────────────────────────────────────────
    training_args = Seq2SeqTrainingArguments(
        output_dir=OUTPUT_DIR,

        # Steps vs epochs
        max_steps=MAX_STEPS if MAX_STEPS > 0 else -1,
        num_train_epochs=NUM_EPOCHS if MAX_STEPS <= 0 else 1,

        per_device_train_batch_size=BATCH_SIZE,
        per_device_eval_batch_size=BATCH_SIZE,
        gradient_accumulation_steps=GRAD_ACCUM_STEPS,

        learning_rate=LEARNING_RATE,
        warmup_steps=WARMUP_STEPS,
        lr_scheduler_type="cosine",

        fp16=FP16,

        # Evaluation & saving
        eval_strategy="steps",
        eval_steps=50,
        save_strategy="steps",
        save_steps=50,
        load_best_model_at_end=True,
        metric_for_best_model="wer",
        greater_is_better=False,

        # Generation settings for eval
        predict_with_generate=True,
        generation_max_length=225,

        logging_steps=10,
        report_to="none",         # change to "tensorboard" if you want TB logs
        push_to_hub=False,
    )

    # ── Trainer ───────────────────────────────────────────────────────────────
    trainer = Seq2SeqTrainer(
        model=model,
        args=training_args,
        train_dataset=ds["train"],
        eval_dataset=ds["validation"],
        data_collator=data_collator,
        compute_metrics=compute_metrics,
    )

    # ── Train ─────────────────────────────────────────────────────────────────
    print("Starting training …")
    trainer.train()

    # ── Save final model & processor ──────────────────────────────────────────
    print(f"Saving fine-tuned model to '{OUTPUT_DIR}' …")
    trainer.save_model(OUTPUT_DIR)
    processor.save_pretrained(OUTPUT_DIR)
    print("Training complete ✓")


if __name__ == "__main__":
    main()

Loading dataset from Hugging Face Hub …
  Train size : 200
  Valid size : 50
Loading model 'openai/whisper-small' …


Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

Starting training …


/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Step,Training Loss,Validation Loss,Wer
50,0.811358,0.482527,6.970000
100,0.166465,0.280662,7.860000
150,0.012527,0.215459,7.350000
200,0.009007,0.219045,7.220000


...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['proj_out.weight'].


Saving fine-tuned model to './whisper-small-librispeech' …


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Training complete ✓
